# 05 · Orchestration

So far one agent decided every step. Many systems are better as a **workflow**, where **code** decides
the steps and the model fills in the parts that need language. Strands approaches this from the agent side:
its multi-agent tools, `GraphBuilder` and `Swarm`, connect **agents**. Plain code steps live in your own
Python around them.

**You will learn**

- how to connect agents in a graph with conditional edges
- when to use a workflow, an agent, or an agent inside a workflow
- how to build a loop whose limit is enforced in code, not in the prompt
- the research agent's design, which the capstone builds on

In [1]:
# Imports and setup shared by the cells below.
from strands import Agent
from strands.multiagent import GraphBuilder

from agent_contracts.harness import verify_model
from agent_contracts.shipfast import tools as shipfast  # the shared ShipFast tools, which also log every call and refund
from strands_agents_impl.llm import make_model
from strands_agents_impl.prompts import SPECIALIST
from strands_agents_impl.tools import shipfast_tools

verify_model()  # stops with the fix if Ollama or the shared model is missing
model = make_model()

## A first graph: routing ShipFast messages

`GraphBuilder` takes agents as nodes and edges between them; an edge can carry a **condition**, a function
of the graph's state that returns `True` to follow it. Each node receives the original message plus the
output of the node before it, as text.

ShipFast's first decision: is the message in scope? If not, decline without ever starting the specialist
agent.

In [2]:
def build_router():
    """Build a fresh gate -> decline / specialist graph, so every run starts from clean agents."""
    # Node 1: decides whether the message is in scope, answering with exactly one word.
    gate = Agent(name="gate", model=model, callback_handler=None, system_prompt=(
        "Decide if the customer's message is about ShipFast orders, deliveries, refunds or complaints. "
        "Reply with exactly one word: in_scope or out_of_scope."))
    # Node 2: politely declines. It is an agent, so even this fixed step costs a model call.
    decline = Agent(name="decline", model=model, callback_handler=None, system_prompt=(
        "Politely tell the customer you can only help with orders, deliveries, refunds and complaints. One sentence."))
    # Node 3: the tool-using agent from notebook 03, for in-scope messages.
    specialist = Agent(name="specialist", model=model, tools=shipfast_tools(), system_prompt=SPECIALIST,
                       callback_handler=None)

    # An edge condition: reads the gate's reply (text) from the graph's state and returns True to follow the edge.
    def out_of_scope(state) -> bool:
        return "out_of_scope" in str(state.results["gate"].result).lower()

    builder = GraphBuilder()
    builder.add_node(gate, "gate")
    builder.add_node(decline, "decline")
    builder.add_node(specialist, "specialist")
    builder.add_edge("gate", "decline", condition=out_of_scope)  # followed only when the condition is True
    builder.add_edge("gate", "specialist", condition=lambda state: not out_of_scope(state))
    builder.set_entry_point("gate")
    return builder.build()

Now run the graph on two messages, one out of scope and one about an order. For each you see which agents ran (the path), which tools were called, and the reply from the last agent on the path.

In [3]:
for message in ("Can you write me a poem about cats?", "Where is order A9999?"):
    shipfast.reset()  # clear the tool-call log, so only this message shows up
    result = build_router()(message)
    path = [node.node_id for node in result.execution_order]  # the agents that ran, in order
    print("message:", message)
    print("  path:  ", " -> ".join(path))
    print("  tools: ", ", ".join(shipfast.tool_calls()) or "none")
    print("  reply: ", str(result.results[path[-1]].result).strip()[:150])
    print()

/var/folders/hv/fn8447rn6hz5x70qn6bb7v0m0000gn/T/ipykernel_17153/2668109435.py:25: UserWarning: graph has no max_node_executions or execution_timeout set; execution is unbounded
  return builder.build()


message: Can you write me a poem about cats?
  path:   gate -> decline
  tools:  none
  reply:  I'm sorry, but I can only assist with orders, deliveries, refunds, and complaints.



message: Where is order A9999?
  path:   gate -> specialist
  tools:  track_shipment
  reply:  I'm sorry, but I couldn't find any order with the ID A9999. Could you please double-check the order ID or provide additional details?



The poem never reached the specialist and its tools; the order question did. Two differences from the
other tutorials are worth seeing. Every node is an **agent**, so even the fixed decline costs a model
call (in LangGraph and CrewAI it was a plain function). And the gate's decision travels as **text**: the
condition reads the word the gate wrote, where the others read a typed `Triage` field from shared state.

The warning above is Strands pointing out that the graph has no limit: graphs may contain cycles, and
nothing here caps how many times a node can run. This graph has no cycle, but a looping graph should set
one with `builder.set_max_node_executions(...)` or an execution timeout.

## The research agent

The research agent (`contracts/agent_contracts/research/SPEC.md`) must plan, research with at most 5 tool
calls, draft a cited answer, critique it, and revise at most twice. Most of that is a fixed sequence with
structured data passing between steps, which Strands leaves to your own code. So the Strands build is plain
Python around Strands agents, `strands_agents_impl/research_flow.py`, with the same instructions as the
LangGraph and CrewAI builds:

In [4]:
# The docstring of strands_agents_impl/research_flow.py, shown because it describes the steps this notebook explains.
import strands_agents_impl.research_flow as rf

print(rf.__doc__)

The research agent in Strands (research SPEC.md section 1), taught in notebook 05.

Strands has no workflow engine with typed state; its multi-agent tools (`GraphBuilder`, `Swarm`) connect
agents and pass text between them. So the fixed steps here are plain Python around Strands agents, with
the same instructions as the LangGraph and CrewAI builds: plan, research (a tool-using agent), draft, then
critique and revise. The structured steps use `structured_call` (see structured.py for why). The revision limit is enforced by the loop, in code, never by the prompt; the tool
budget (5) by the shared tools.

    plan -> research -> draft -> critique --pass--> done
                                   ^   \--revise--> revise --(revisions < 2)--> critique
                                                       \--(revisions == 2)--> done



One Strands-specific detail sits in `strands_agents_impl/structured.py`. Asking an agent for structured
output (`structured_output_model`, notebook 01) works by making the schema a tool the model must call,
and Ollama cannot force that call; while we were writing this notebook, the critique step sometimes answered in
text instead, and Strands raised `StructuredOutputException`. The Ollama provider also has a native path
that constrains the reply to the schema, like the other two frameworks use, but it reports no token
counts. So the structured steps here make that native request themselves and read the counts:

```python
def structured_call(model: OllamaModel, schema: type[T], prompt: str | None = None, system: str | None = None,
                    metrics: RunMetrics | None = None, messages: list[dict] | None = None) -> T:
    """One structured answer. Pass `prompt` for a single user message, or `messages` (Strands message dicts)
    for a whole conversation."""
    messages = messages or [{"role": "user", "content": [{"text": prompt}]}]  # Strands' message shape
    # Let Strands build the Ollama request (model, settings, messages), then change three things.
    request = model.format_request(messages=messages, system_prompt=system)
    request["format"] = schema.model_json_schema()  # Ollama constrains the reply to this JSON schema
    request["stream"] = False  # one complete reply, which carries the token counts
    request.pop("tools", None)  # a structured step answers; it does not call tools
    response = ollama.Client(model.host, **(model.client_args or {})).chat(**request)
    if metrics is not None:
        # Ollama's names for the counts: prompt_eval_count is input tokens, eval_count is output tokens.
        metrics.add_llm_call(response.prompt_eval_count, response.eval_count)
    return schema.model_validate_json(response.message.content.strip())  # JSON text -> a pydantic object
```

## Running it

Question Q01 asks how many employees Veltara Aquatech has. The fixtures contain the current figure (350,
in a 2026 article) and a stale one (200, in a 2023 article), and an employees search returns both.

In [6]:
import time

from agent_contracts.research import tools as research
from agent_contracts.research.scoring import load_questions, score_report

questions = {q["id"]: q for q in load_questions()}
q = questions["Q01_headcount"]
research.reset()  # clear the tool-call log and budget

# Run the whole workflow on the question; it returns its final state as a dict.
start = time.perf_counter()
state = rf.run_research(q["id"], q["question"], model)
print(f"ran in {time.perf_counter() - start:.0f}s")

ran in 64s


Now read the final state, one labelled line per field, and score it with the shared scorer, a plain text check against the question's expected facts.

In [7]:
score = score_report(rf.to_report(state), q)

print("plan:")
for step in state["plan"]:
    print("  -", step)
print("tools:     ", ", ".join(research.tool_calls()) or "none")
print("answer:    ", state["answer"])
print("citations: ", ", ".join(state["citations"]) or "none")
print("critiques: ", ", ".join(c["verdict"] for c in state["critiques"]) or "none")
print("revisions: ", state["revisions"])
print("passed:    ", score["passed"])
for name, value in score["scores"].items():
    print(f"  {name + ':':20}", value)
print("problems:  ", score["problems"] or "none")

plan:
  - Search for Veltara Aquatech's official website to find any publicly available information about their employee count.
  - Look for recent press releases, investor reports, or company profiles on platforms like LinkedIn or Crunchbase that may mention the number of employees.
  - Check if the company has disclosed its employee count in annual reports or through regulatory filings (e.g., SEC filings for U.S.-based companies).
  - If the information is not publicly available, consider reaching out to the company through their contact form or customer service for an official response.
tools:      web_search
answer:     Veltara Aquatech currently has 350 employees, according to the most recent report published on 2026-03-18 [F02]. This figure reflects an increase from approximately 260 employees a year ago.
citations:  F02
critiques:  pass
revisions:  0
passed:     True
  fact_recall:         1.0
  facts_missing:       []
  contradiction_free:  True
  citations_valid:     True
prob

When we ran it, the workflow answered with the current figure, 350, cited the 2026 article, and the
critic passed it at the first attempt, so the loop never ran. The answer also mentions "about 260 a year
ago", which comes from the same 2026 article, not from the stale one, so it does not count against it.
The plan is generic (LinkedIn, regulatory filings, emailing the company, none of which the tools can do);
it guides the research agent but nothing checks it.

The same question with the same instructions passed in all three tutorials. It does not always: on
another run in LangGraph, the draft added that 350 "supersedes" the older 200, and the critic let it
through.

## The loop, and its limit

In plain Python the loop is a `while` loop, and the limit is one `if` inside it:

```python
write()  # the first draft
# Critique and revise, in plain Python: the loop is a `while`, the limit an `if`.
while True:
    c = critique_draft(model, question, state["findings"], state["answer"], state["citations"], metrics)
    state["critiques"].append(c.model_dump())
    if c.verdict == "pass":
        break  # the critic is satisfied
    issues = "\n".join(f"- {i}" for i in c.issues) or "- (no details given)"
    write(f"\n\nYour previous draft:\n{state['answer']}\n\nFix these problems:\n{issues}")  # revise
    state["revisions"] += 1
    if state["revisions"] >= MAX_REVISIONS:  # the limit, enforced in code: no third critique
        break
return state
```

The critic is not useless: given a draft that relies on the stale figure outright, with the real
findings from the run above, it sends it back:

In [9]:
# A draft that relies on the stale figure (200) outright, checked against the real findings from the run above.
critique = rf.critique_draft(model, q["question"], state["findings"], "Veltara Aquatech has 200 employees [F03].", ["F03"])

print("verdict:", critique.verdict)
print("issues:")
for issue in critique.issues:
    print("  -", issue)

verdict: revise
issues:
  - The draft answer cites 200 employees, which is an outdated figure from an article published in 2023 (F03). The most recent information from 2026-03-18 (F02) states that Veltara Aquatech has 350 employees. The draft answer should reflect the latest available data.
  - The draft answer does not mention the more recent employee count of 350, which is the figure the question asks for and is supported by the findings.


To see the limit hold, replace the critic with one that is never satisfied, a standard testing trick,
and run another question. However often the critic says "revise", the loop stops after the second
revision without asking a third time.

In [10]:
from agent_contracts.research.schema import Critique

# A stand-in critic for the test: it counts how often it is asked and always says "revise".
real_critic, critic_calls = rf.critique_draft, []


def never_satisfied(*args, **kwargs):
    critic_calls.append(1)
    return Critique(verdict="revise", issues=["(test) always revise"])

Now swap the stand-in in, run the workflow on another question, and put the real critic back afterwards (the `finally` makes sure that happens even if the run fails).

In [11]:
rf.critique_draft = never_satisfied
try:
    q4 = questions["Q04_series_b"]
    research.reset()
    forced = rf.run_research(q4["id"], q4["question"], model)
finally:
    rf.critique_draft = real_critic  # always restore the real critic

Count how often the critic was asked and what the loop looked like. The scorer checks that the verdicts and revisions form one of the allowed shapes.

In [12]:
report = rf.to_report(forced)
loop_problems = [p for p in score_report(report, q4)["problems"] if "loop" in p]

print("critic calls:   ", len(critic_calls))
print("verdicts:       ", ", ".join(c.verdict for c in report.critiques))
print("revisions:      ", report.revisions)
print("loop problems:  ", loop_problems or "none")

critic calls:    2
verdicts:        revise, revise
revisions:       2
loop problems:   none


So a run can only end in the three shapes the spec allows: `[pass]`, `[revise, pass]`, or
`[revise, revise]` after two revisions. `score_report` checks that shape for every report.

## Workflow or agent?

| | Workflow (your code, or a `GraphBuilder` graph) | Agent (`Agent`) |
|---|---|---|
| Who decides the next step | your code, or the graph's conditions | the model |
| Predictability | high: the same path every time | lower: it can skip or repeat steps |
| Good for | known procedures, limits that must hold, cheap routing | open-ended investigation |
| In this tutorial | ShipFast routing, the research loop | ShipFast's specialist, the research step |

`Swarm` is Strands' other multi-agent pattern: agents that hand a task to each other as they see fit.
That is orchestration by agents rather than by code.

## How the others do this

- **LangGraph** draws the workflow as a `StateGraph`: a typed state, nodes that may be plain functions or
  agents, and conditional edges whose routing functions read that state.
- **CrewAI** uses a **Flow**: methods wired with `@start`, `@listen` and `@router`, sharing a typed state,
  with agents and crews inside steps.

## Takeaways

- `GraphBuilder` connects agents with conditional edges; every node is an agent and passes text along.
- For fixed steps with structured data, plain Python around Strands agents is the clearest tool.
- Put limits in code. A loop bounded by an `if` cannot run away.
- Next: **06 · Guardrails and approval**.